# CUDA Streams and Overlap

This notebook explores how CUDA operations are ordered, queued, and potentially executed concurrently. The first experiment verifies that operations submitted to the same
CUDA stream execute in submission order.

## Questions

1. Can the CPU submit operation B before operation A finishes?
2. What ordering guarantees does one CUDA stream provide?
3. Can two GPU operations execute concurrently?
4. When do multiple streams improve throughput?
5. How are dependencies expressed across streams?

## Mental model

A CUDA stream is an ordered queue of GPU work. If operations A and B are submitted to the same stream:

#### CPU:

submit A → submit B → continue running

#### CUDA stream:

A → B

The CPU does not need to wait for A before submitting B. However, the GPU will not begin B until A has completed because both operations belong to the same ordered stream.

In [2]:
%pip install -q "modal>=1.5,<2"


[notice] A new release of pip is available: 23.2.1 -> 26.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys
import modal

LOCAL_PYTHON = f"{sys.version_info.major}.{sys.version_info.minor}"

app = modal.App("gpu-inference-systems-cuda-streams")

image = (
    modal.Image.debian_slim(python_version=LOCAL_PYTHON)
    .pip_install(
        "torch",
        "numpy",
    )
)

In [7]:
# Same-stream experiment

@app.function(
    image=image,
    gpu="L4",
    timeout=300
)
def measure_same_stream_ordering() -> dict[str, float | str]:
    import time
    import torch

    size = 4096

    x1 = torch.randn(size, size, device="cuda")
    y1 = torch.randn(size, size, device="cuda")
    z1 = torch.zeros(size, size, device="cuda")

    x2 = torch.randn(size, size, device="cuda")
    y2 = torch.randn(size, size, device="cuda")
    z2 = torch.randn(size, size, device="cuda")

    for _ in range(3):
        torch.mm(x1, y1, out=z1)
        torch.mm(x2, y2, out=z2)

    start_event = torch.cuda.Event(enable_timing=True)
    after_a_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    cpu_start = time.perf_counter()
    start_event.record()

    # Operation A
    torch.mm(x1, y1, out=z1)
    
    # This event is queued after A in the same stream.
    after_a_event.record()

    # Operation B
    torch.mm(x2, y2, out=z2)
    
    end_event.record()
    cpu_submission_ms = (time.perf_counter() - cpu_start) * 1000
    end_event.synchronize()
    
    operation_a_ms = start_event.elapsed_time(after_a_event)
    operation_b_ms = after_a_event.elapsed_time(end_event)
    total_gpu_ms = start_event.elapsed_time(end_event)

    return {
        "gpu_name": torch.cuda.get_device_name(0),
        "cpu_submission_ms": cpu_submission_ms,
        "operation_a_ms": operation_a_ms,
        "operation_b_ms": operation_b_ms,
        "total_gpu_ms": total_gpu_ms,
    }

In [8]:
with modal.enable_output():
    with app.run():
        same_stream_result = measure_same_stream_ordering.remote()

same_stream_result

✓ Initialized. View run at 
https://modal.com/apps/vishalkt07/main/ap-l7qowmmrRNPb7BLqoPWkqN
⠋ Initializing...
⠴ Creating objects.....
└── 🔨 Created function measure_same_stream_ordering.
✓ Created objects.
└── 🔨 Created function measure_same_stream_ordering.
⠹ Worker assigned... View app at 
⠴ Running... View app at qoPWkqN
⠇ Loading images (1 containers initializing)... View app at 
⠹ Loading images (1 containers initializing)... View app at 
⠴ Loading images (1 containers initializing)... View app at 
⠇ Loading images (1 containers initializing)... View app at 
⠙ Loading images (1 containers initializing)... View app at 
⠼ Loading images (1 containers initializing)... View app at 
⠇ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 
⠧ Running (1/1 containers active)... View app at 
⠋ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 
⠧ Runni

{'gpu_name': 'NVIDIA L4',
 'cpu_submission_ms': 0.19628699999962862,
 'operation_a_ms': 11.673600196838379,
 'operation_b_ms': 11.65824031829834,
 'total_gpu_ms': 23.33184051513672}

In [9]:
# Verify the timing relationship
measured_sum = (
    same_stream_result["operation_a_ms"]
    + same_stream_result["operation_b_ms"]
)

difference_ms = abs(
    same_stream_result["total_gpu_ms"] - measured_sum
)

print("A + B:", measured_sum)
print("Total:", same_stream_result["total_gpu_ms"])
print("Difference:", difference_ms)

assert difference_ms < 1.0

print("PASS: operations in the same stream executed sequentially")

A + B: 23.33184051513672
Total: 23.33184051513672
Difference: 0.0
PASS: operations in the same stream executed sequentially


## Observation

The CPU submitted two matrix multiplications and three event markers in
approximately 0.196 ms.

The GPU took approximately:

- 11.67 ms for operation A
- 11.66 ms for operation B
- 23.33 ms in total

The total GPU duration exactly equaled the sum of the two operation
intervals because all three event timestamps belonged to the same CUDA
stream.

This confirms that the CPU can enqueue operation B before operation A
finishes, while the stream still guarantees that B executes only after A.

## Prediction: two independent streams

Operation A and operation B will be placed in separate CUDA streams.

I expect:

- The CPU submission time to remain small.
- The two operations to become eligible for concurrent execution.
- The total elapsed time to be less than A time plus B time if the GPU
  has enough unused resources to overlap them.

However, overlap is not guaranteed. A single 4096×4096 matrix
multiplication may already use most of the GPU.

In [16]:
@app.function(
    image=image,
    gpu="L4",
    timeout=300,
)
def measure_two_streams() -> dict[str, float | str]:
    import time
    import torch

    size = 4096

    x1 = torch.randn(size, size, device="cuda")
    y1 = torch.randn(size, size, device="cuda")
    z1 = torch.empty(size, size, device="cuda")

    x2 = torch.randn(size, size, device="cuda")
    y2 = torch.randn(size, size, device="cuda")
    z2 = torch.empty(size, size, device="cuda")

    stream_a = torch.cuda.Stream()
    stream_b = torch.cuda.Stream()

    # Warm up both operations.
    for _ in range(3):
        torch.mm(x1, y1, out=z1)
        torch.mm(x2, y2, out=z2)
    torch.cuda.synchronize()

    common_start = torch.cuda.Event(enable_timing=True)
    start_a = torch.cuda.Event(enable_timing=True)
    end_a = torch.cuda.Event(enable_timing=True)
    start_b = torch.cuda.Event(enable_timing=True)
    end_b = torch.cuda.Event(enable_timing=True)
    final_event = torch.cuda.Event(enable_timing=True)
    cpu_start = time.perf_counter()

    # Record the common starting point on the default stream.
    common_start.record()

    # Make both streams wait until the start marker is reached.
    stream_a.wait_event(common_start)
    stream_b.wait_event(common_start)

    with torch.cuda.stream(stream_a):
        start_a.record()
        torch.mm(x1, y1, out=z1)
        end_a.record()

    with torch.cuda.stream(stream_b):
        start_b.record()
        torch.mm(x2, y2, out=z2)
        end_b.record()

    cpu_submission_ms = (time.perf_counter() - cpu_start) * 1000
    
    # The default stream waits for both independent streams.
    torch.cuda.current_stream().wait_event(end_a)
    torch.cuda.current_stream().wait_event(end_b)

    final_event.record()
    final_event.synchronize()

    operation_a_ms = start_a.elapsed_time(end_a)
    operation_b_ms = start_b.elapsed_time(end_b)
    total_gpu_ms = common_start.elapsed_time(final_event)
    sequential_estimate_ms = operation_a_ms + operation_b_ms
    estimated_overlap_ms = max(0.0, sequential_estimate_ms - total_gpu_ms)

    return {
        "gpu_name": torch.cuda.get_device_name(0),
        "cpu_submission_ms": cpu_submission_ms,
        "operation_a_ms": operation_a_ms,
        "operation_b_ms": operation_b_ms,
        "sequential_estimate_ms": sequential_estimate_ms,
        "total_gpu_ms": total_gpu_ms,
        "estimated_overlap_ms": estimated_overlap_ms,
    }

In [18]:
with modal.enable_output():
    with app.run():
        two_stream_result = measure_two_streams.remote()

two_stream_result

✓ Initialized. View run at 
https://modal.com/apps/vishalkt07/main/ap-5yVBkiee3kg8LUPzPCkNXI
⠋ Initializing...
⠴ Creating objects.....
├── 🔨 Created function measure_same_stream_ordering.
└── 🔨 Created function measure_two_streams.
✓ Created objects.
├── 🔨 Created function measure_same_stream_ordering.
└── 🔨 Created function measure_two_streams.
⠼ Worker assigned... View app at 
⠧ Worker assigned... View app at 
⠋ Loading images (1 containers initializing)... View app at 
⠸ Loading images (1 containers initializing)... View app at 
⠦ Loading images (1 containers initializing)... View app at 
⠋ Loading images (1 containers initializing)... View app at 
⠸ Loading images (1 containers initializing)... View app at 
⠦ Loading images (1 containers initializing)... View app at 
⠏ Loading images (1 containers initializing)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠦ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 
⠹ Runni

{'gpu_name': 'NVIDIA L4',
 'cpu_submission_ms': 0.9689929999998625,
 'operation_a_ms': 11.428863525390625,
 'operation_b_ms': 21.84294319152832,
 'sequential_estimate_ms': 33.271806716918945,
 'total_gpu_ms': 22.43788719177246,
 'estimated_overlap_ms': 10.833919525146484}

## Observation

The two matrix multiplications overlapped for approximately 11 ms. However, total elapsed time was approximately 23.38 ms. Operation A completed in approximately 11.95 ms, while operation B took approximately 22.42 ms. This suggests that the concurrent kernels competed for GPU resources and one operation progressed more slowly.

The experiment demonstrates that concurrency and throughput improvement are different concepts. Multiple streams make concurrent execution possible, but they only improve performance when the operations leave complementary GPU resources available.

## CUDA Events, Stream Dependencies, and Synchronization

Before continuing, it is useful to distinguish CUDA events, stream waits, and CPU synchronization.

### CUDA streams
A CUDA stream is an ordered queue of GPU commands. Operations submitted to the same stream execute in submission order:

```text
Stream A:

Operation 1 → Operation 2 → Operation 3
```

The CPU can enqueue all three operations without waiting for each operation to finish. Outside a custom stream context, PyTorch normally submits CUDA work to the
current stream, which is usually the default stream:

```python
torch.cuda.current_stream()
torch.cuda.default_stream()
```

Inside a stream context:

```python
with torch.cuda.stream(stream_a):
    torch.mm(x, y)
```

`stream_a` temporarily becomes the current stream. After leaving the block,
the previous stream becomes current again.

---

### CUDA events

A CUDA event is a device-visible synchronization marker.

Calling:

```python
event.record()
```
inserts an event marker into the current stream.

For example:

```python
torch.mm(x, y)
event.record()
```

creates the following stream order:

```text
Matrix multiplication → Event marker
```

`event.record()` returns asynchronously on the CPU, but the recorded event completes only when the GPU reaches that point in the stream.

```text
CPU:

- Submit matrix multiplication
- Submit event marker
- Continue executing

GPU stream:

[Matrix multiplication] → [Event becomes complete]
```

An event is not permanently owned by the stream on which it was recorded. The recording stream produces the event's completion state, but other streams
can use the same event as a dependency.

---

### Cross-stream dependencies

Suppose `common_start` is recorded on the default stream:

```python
common_start.record()
```

Conceptually:

```text
Default stream:

[Earlier work] → [common_start completes]
```

Another stream can wait for that event:

```python
stream_a.wait_event(common_start)
```

This adds a dependency to `stream_a`:

```text
Stream A:

[Wait for common_start] → [Future work]
```

The event connects the producer stream and consumer stream:

```text
Default stream                         Stream A

[Earlier work]
      |
[common_start completes] ------------> [Wait is satisfied]
                                              |
                                        [Future work]
```

The CUDA runtime and GPU scheduler know that both commands refer to the same underlying event object. When the default stream reaches the event marker, the event becomes complete. Work after the wait in `stream_a` then becomes eligible to execute. The CPU does not continuously check the event, and a GPU kernel does not need to spin while checking it. At the CUDA programming-model level, this is a scheduling dependency between two streams.

---

### Does `wait_event()` block the CPU?

No.

```python
stream_a.wait_event(common_start)
```

returns after submitting the dependency.

It means:

> Future work in `stream_a` cannot execute until `common_start` completes.

It does not mean:

> Stop the CPU until `common_start` completes.

For example:

```python
stream_a.wait_event(common_start)

with torch.cuda.stream(stream_a):
    torch.mm(x1, y1, out=z1)

print("The CPU can reach this before the GPU finishes")
```

The resulting queues are approximately:

```text
Default stream:

[common_start]

Stream A:

[Wait for common_start] → [Matrix multiplication]
```

The CPU can submit both queues before the GPU executes them.

---

### `record()`, `wait_event()`, and `synchronize()`

These operations create different kinds of dependencies.

#### `event.record()`

```python
event.record()
```

Inserts an event marker into a stream.

```text
Stream work → Event marker
```

The CPU does not block.

#### `stream.wait_event(event)`

```python
stream_a.wait_event(event)
```

Makes future work in one GPU stream depend on an event produced by another
stream.

```text
Producer stream → Event → Consumer stream
```

The CPU does not block.

#### `event.synchronize()`

```python
event.synchronize()
```

Makes the CPU wait until that event becomes complete.

```text
GPU event → CPU may continue
```

The CPU blocks.

#### `stream.synchronize()`

```python
stream_a.synchronize()
```

Makes the CPU wait until previously submitted work in `stream_a` completes. The CPU blocks, but only for the selected stream.

#### `torch.cuda.synchronize()`

```python
torch.cuda.synchronize()
```

Makes the CPU wait until previously submitted CUDA work on the selected device completes across its streams. This is a device-wide synchronization boundary and is broader than waiting for one event or one stream. A compact mental model is:

```text
event.record()
    Place a checkpoint in a GPU stream.

stream.wait_event(event)
    Make one GPU stream depend on that checkpoint.

event.synchronize()
    Make the CPU wait for that checkpoint.

stream.synchronize()
    Make the CPU wait for one stream.

torch.cuda.synchronize()
    Make the CPU wait for the device's submitted work.
```

---

### Why use `common_start`?

In the two-stream experiment, `common_start` was recorded on the default stream:

```python
common_start.record()

stream_a.wait_event(common_start)
stream_b.wait_event(common_start)
```

This created a common release point:

```text
                             ┌──> Stream A
Default stream               |
[common_start completes] ----|
                             |
                             └──> Stream B
```

After `common_start` completes, both streams become eligible to execute their
future work.

```text
                            ┌──> start_a → Operation A → end_a
common_start completes -----|
                            └──> start_b → Operation B → end_b
```

`common_start` is not required merely to launch work in two streams.

It was used because it provides:

1. An explicit dependency instead of relying on implicit default-stream
   behavior.
2. A common timing origin for both streams.
3. A clean point after which both operations are allowed to begin.

Because there was little work before `common_start` in the default stream, the
wait was likely very short. Its purpose was coordination and measurement, not
to intentionally delay either operation.

---

### Why return to the default stream at the end?

Each worker stream recorded an event after its operation:

```text
Stream A:

Operation A → end_a

Stream B:

Operation B → end_b
```

The default stream then waited for both events:

```python
default_stream.wait_event(end_a)
default_stream.wait_event(end_b)
final_event.record()
```

This creates a join point:

```text
end_a completes ----┐
                    ├──> final_event
end_b completes ----┘
```

`final_event` cannot complete until both operations have finished.

Finally:

```python
final_event.synchronize()
```

makes the CPU wait for the final join point.

The complete experiment follows a fork-and-join structure:

```text
                         ┌──> Operation A ──> end_a ──┐
common_start ────────────|                            ├──> final_event
                         └──> Operation B ──> end_b ──┘
```

Most of this coordination occurs entirely on the GPU.

The CPU becomes blocked only at:

```python
final_event.synchronize()
```

---

### Key distinction

```text
stream.wait_event(event)
```

means:

```text
GPU stream waits for GPU event
```

while:

```text
event.synchronize()
```

means:

```text
CPU waits for GPU event
```

Events therefore support both device-side coordination and host-side
synchronization, depending on how they are used.

### Final mental model

```text
event.record(producer_stream)
    The producer stream will publish completion when it reaches the event.

consumer_stream.wait_event(event)
    Future work in the consumer stream depends on that completion.

event.synchronize()
    The CPU depends on that completion.
```

An event can therefore act as a bridge between streams:

```text
Producer stream → Event → Consumer stream
```

or as a bridge back to the CPU:

```text
GPU stream → Event → CPU
```


## Experiment: Smaller operations across multiple streams

The previous experiment showed that two large matrix multiplications could overlap, but total throughput did not improve. Each large operation already consumed substantial GPU resources. Running both concurrently caused contention and slowed one of them down. In this experiment, I will execute a sequence of smaller matrix multiplications.

I will compare:

1. All operations submitted sequentially to one stream.
2. The same number of operations divided between two streams.

If the smaller operations do not fully utilize the GPU, two streams may allow the GPU to execute work from both queues concurrently and improve total throughput.

In [23]:
@app.function(
    image=image,
    gpu="L4",
    timeout=300,
)
def compare_small_operations(
    size: int = 1024,
    repetitions_per_stream: int = 100,
) -> dict[str, float | int | str]:
    import torch

    dtype = torch.float16

    x1 = torch.randn(size, size, device="cuda", dtype=dtype)
    y1 = torch.randn(size, size, device="cuda", dtype=dtype)
    z1 = torch.empty(size, size, device="cuda", dtype=dtype)

    x2 = torch.randn(size, size, device="cuda", dtype=dtype)
    y2 = torch.randn(size, size, device="cuda", dtype=dtype)
    z2 = torch.empty(size, size, device="cuda", dtype=dtype)

    stream_a = torch.cuda.Stream()
    stream_b = torch.cuda.Stream()
    default_stream = torch.cuda.current_stream()

    # Warm up the exact operations used in the benchmark.
    for _ in range(10):
        torch.mm(x1, y1, out=z1)
        torch.mm(x2, y2, out=z2)

    torch.cuda.synchronize()

    single_start = torch.cuda.Event(enable_timing=True)
    single_end = torch.cuda.Event(enable_timing=True)

    single_start.record()

    for _ in range(repetitions_per_stream):
        torch.mm(x1, y1, out=z1)

    for _ in range(repetitions_per_stream):
        torch.mm(x2, y2, out=z2)

    single_end.record()
    single_end.synchronize()

    single_stream_ms = single_start.elapsed_time(single_end)
    common_start = torch.cuda.Event(enable_timing=True)

    start_a = torch.cuda.Event(enable_timing=True)
    end_a = torch.cuda.Event(enable_timing=True)

    start_b = torch.cuda.Event(enable_timing=True)
    end_b = torch.cuda.Event(enable_timing=True)

    final_event = torch.cuda.Event(enable_timing=True)

    common_start.record()

    stream_a.wait_event(common_start)
    stream_b.wait_event(common_start)

    with torch.cuda.stream(stream_a):
        start_a.record()

        for _ in range(repetitions_per_stream):
            torch.mm(x1, y1, out=z1)

        end_a.record()

    with torch.cuda.stream(stream_b):
        start_b.record()

        for _ in range(repetitions_per_stream):
            torch.mm(x2, y2, out=z2)

        end_b.record()

    # Join both worker streams back onto the default stream.
    default_stream.wait_event(end_a)
    default_stream.wait_event(end_b)

    final_event.record()
    final_event.synchronize()

    stream_a_ms = start_a.elapsed_time(end_a)
    stream_b_ms = start_b.elapsed_time(end_b)
    two_stream_ms = common_start.elapsed_time(final_event)

    total_operations = repetitions_per_stream * 2

    single_throughput = total_operations / (single_stream_ms / 1000)
    two_stream_throughput = total_operations / (two_stream_ms / 1000)

    speedup = single_stream_ms / two_stream_ms

    return {
        "gpu_name": torch.cuda.get_device_name(0),
        "matrix_size": size,
        "repetitions_per_stream": repetitions_per_stream,
        "total_operations": total_operations,
        "single_stream_ms": single_stream_ms,
        "stream_a_ms": stream_a_ms,
        "stream_b_ms": stream_b_ms,
        "two_stream_ms": two_stream_ms,
        "single_stream_ops_per_second": single_throughput,
        "two_stream_ops_per_second": two_stream_throughput,
        "speedup": speedup,
    }

In [21]:
with modal.enable_output():
    with app.run():
        small_operation_result = compare_small_operations.remote(
            size=1024,
            repetitions_per_stream=100,
        )

small_operation_result

✓ Initialized. View run at 
https://modal.com/apps/vishalkt07/main/ap-MJyD0mgn9vS4jjBv5J5fVJ
⠋ Initializing...
⠦ Creating objects.....
├── ⠋ Creating function measure_same_stream_ordering...
├── ⠋ Creating function measure_two_streams...
⠧ Creating objects...all_operations...
├── 🔨 Created function measure_same_stream_ordering.
├── 🔨 Created function measure_two_streams.
└── 🔨 Created function compare_small_operations.
✓ Created objects.A
├── 🔨 Created function measure_same_stream_ordering.
├── 🔨 Created function measure_two_streams.
└── 🔨 Created function compare_small_operations.
⠹ Worker assigned... View app at 
⠴ Worker assigned... View app at 
⠇ Loading images (1 containers initializing)... View app at 
⠹ Loading images (1 containers initializing)... View app at 
⠴ Loading images (1 containers initializing)... View app at 
⠇ Loading images (1 containers initializing)... View app at 
⠙ Loading images (1 containers initializing)... View app at 
⠼ Loading images (1 containers initial

{'gpu_name': 'NVIDIA L4',
 'matrix_size': 1024,
 'repetitions_per_stream': 100,
 'total_operations': 200,
 'single_stream_ms': 7.946239948272705,
 'stream_a_ms': 7.6769280433654785,
 'stream_b_ms': 5.443583965301514,
 'two_stream_ms': 10.052607536315918,
 'single_stream_ops_per_second': 25169.136761780584,
 'two_stream_ops_per_second': 19895.33554129937,
 'speedup': 0.7904655503128142}

In [22]:
result = small_operation_result

print(
    f"Single-stream time: {result['single_stream_ms']:.2f} ms"
)

print(
    f"Two-stream time:    {result['two_stream_ms']:.2f} ms"
)

print(
    f"Speedup:            {result['speedup']:.2f}x"
)

if result["speedup"] > 1.05:
    print("Multiple streams produced a meaningful throughput improvement.")
elif result["speedup"] < 0.95:
    print("Multiple streams reduced throughput.")
else:
    print("Multiple streams produced little or no throughput improvement.")

Single-stream time: 7.95 ms
Two-stream time:    10.05 ms
Speedup:            0.79x
Multiple streams reduced throughput.


## Observation: Submission order matters

The first two-stream implementation was slower than the single-stream
baseline:

- Single stream: approximately 7.95 ms
- Two streams: approximately 10.05 ms
- Speedup: 0.79x

However, the CPU submitted all operations for stream A before submitting
operations for stream B. The common-start event only prevented operations from running before the common boundary. It did not guarantee that both streams already had work
queued when that boundary completed. As a result, stream A likely began several milliseconds before stream B. The experiment therefore measured both GPU resource contention and host-side
submission skew. To make the comparison fairer, operations should be submitted in an
interleaved order so that both streams receive work throughout the submission
period.

In [26]:
@app.function(
    image=image,
    gpu="L4",
    timeout=300,
)
def compare_small_operations(
    size: int = 1024,
    repetitions_per_stream: int = 100,
) -> dict[str, float | int | str]:
    import torch

    dtype = torch.float16

    x1 = torch.randn(size, size, device="cuda", dtype=dtype)
    y1 = torch.randn(size, size, device="cuda", dtype=dtype)
    z1 = torch.empty(size, size, device="cuda", dtype=dtype)

    x2 = torch.randn(size, size, device="cuda", dtype=dtype)
    y2 = torch.randn(size, size, device="cuda", dtype=dtype)
    z2 = torch.empty(size, size, device="cuda", dtype=dtype)

    stream_a = torch.cuda.Stream()
    stream_b = torch.cuda.Stream()
    default_stream = torch.cuda.current_stream()

    # Warm up the exact operations used in the benchmark.
    for _ in range(10):
        torch.mm(x1, y1, out=z1)
        torch.mm(x2, y2, out=z2)

    torch.cuda.synchronize()

    single_start = torch.cuda.Event(enable_timing=True)
    single_end = torch.cuda.Event(enable_timing=True)

    single_start.record()

    for _ in range(repetitions_per_stream):
        torch.mm(x1, y1, out=z1)

    for _ in range(repetitions_per_stream):
        torch.mm(x2, y2, out=z2)

    single_end.record()
    single_end.synchronize()

    single_stream_ms = single_start.elapsed_time(single_end)
    
    common_start = torch.cuda.Event(enable_timing=True)

    start_a = torch.cuda.Event(enable_timing=True)
    end_a = torch.cuda.Event(enable_timing=True)

    start_b = torch.cuda.Event(enable_timing=True)
    end_b = torch.cuda.Event(enable_timing=True)

    final_event = torch.cuda.Event(enable_timing=True)

    common_start.record()

    stream_a.wait_event(common_start)
    stream_b.wait_event(common_start)

    with torch.cuda.stream(stream_a):
        start_a.record()
    
    with torch.cuda.stream(stream_b):
        start_b.record()

    for _ in range(repetitions_per_stream):
        with torch.cuda.stream(stream_a):
            torch.mm(x1, y1, out=z1)

        with torch.cuda.stream(stream_b):
            torch.mm(x2, y2, out=z2)

    with torch.cuda.stream(stream_a):
        end_a.record()

    with torch.cuda.stream(stream_b):
        end_b.record()

    # Join both worker streams back onto the default stream.
    default_stream.wait_event(end_a)
    default_stream.wait_event(end_b)

    final_event.record()
    final_event.synchronize()

    stream_a_ms = start_a.elapsed_time(end_a)
    stream_b_ms = start_b.elapsed_time(end_b)
    two_stream_ms = common_start.elapsed_time(final_event)

    total_operations = repetitions_per_stream * 2

    single_throughput = total_operations / (single_stream_ms / 1000)
    two_stream_throughput = total_operations / (two_stream_ms / 1000)

    speedup = single_stream_ms / two_stream_ms

    return {
        "gpu_name": torch.cuda.get_device_name(0),
        "matrix_size": size,
        "repetitions_per_stream": repetitions_per_stream,
        "total_operations": total_operations,
        "single_stream_ms": single_stream_ms,
        "stream_a_ms": stream_a_ms,
        "stream_b_ms": stream_b_ms,
        "two_stream_ms": two_stream_ms,
        "single_stream_ops_per_second": single_throughput,
        "two_stream_ops_per_second": two_stream_throughput,
        "speedup": speedup,
    }

In [27]:
with modal.enable_output():
    with app.run():
        small_operation_result = compare_small_operations.remote(
            size=1024,
            repetitions_per_stream=100,
        )

small_operation_result

✓ Initialized. View run at 
https://modal.com/apps/vishalkt07/main/ap-82sfG57ewNqjia9bCMVxOE
⠋ Initializing...
⠦ Creating objects.....
├── 🔨 Created function measure_same_stream_ordering.
├── 🔨 Created function measure_two_streams.
⠦ Creating objects...all_operations...
├── 🔨 Created function measure_same_stream_ordering.
├── 🔨 Created function measure_two_streams.
└── 🔨 Created function compare_small_operations.
✓ Created objects.A
├── 🔨 Created function measure_same_stream_ordering.
├── 🔨 Created function measure_two_streams.
└── 🔨 Created function compare_small_operations.
⠙ Worker assigned... View app at 
⠼ Worker assigned... View app at 
⠧ Worker assigned... View app at 
⠋ Worker assigned... View app at 
⠼ Running... View app at bCMVxOE
⠧ Loading images (1 containers initializing)... View app at 
⠋ Loading images (1 containers initializing)... View app at 
⠸ Loading images (1 containers initializing)... View app at 
⠧ Loading images (1 containers initializing)... View app at 
⠋ Lo

{'gpu_name': 'NVIDIA L4',
 'matrix_size': 1024,
 'repetitions_per_stream': 100,
 'total_operations': 200,
 'single_stream_ms': 7.450623989105225,
 'stream_a_ms': 7.25491189956665,
 'stream_b_ms': 7.242815971374512,
 'two_stream_ms': 7.366559982299805,
 'single_stream_ops_per_second': 26843.389264100926,
 'two_stream_ops_per_second': 27149.714450239357,
 'speedup': 1.0114115689015506}

## Observation: Overlap without meaningful speedup

After interleaving submissions and correctly recording each stream's start
event, the two-stream result was:

- Single-stream time: approximately 7.45 ms
- Two-stream time: approximately 7.37 ms
- Speedup: approximately 1.01x

The result is effectively a tie. A 1% difference is small enough to be
benchmark variance rather than a meaningful performance improvement.

Both streams lasted approximately 7.25 ms, while the total two-stream interval
was approximately 7.37 ms. This indicates that the streams executed during
nearly the same time window.

However, 100 operations in each concurrent stream took nearly as long as 200
operations in the single-stream baseline. The two streams competed for the
same GPU resources, causing each stream to progress more slowly.

This experiment demonstrates:

1. Multiple streams can create concurrent execution.
2. Concurrent execution does not create additional GPU compute capacity.
3. If the kernels compete for the same resources, concurrency may produce
   little or no throughput improvement.
4. Throughput must be evaluated using the total amount of completed work, not
   merely by observing that stream timelines overlap.

SyntaxError: invalid syntax (2198802570.py, line 1)